In [1]:
# Memory safety: cap this kernel to the RAM free right now so an out-of-memory
# feature build fails with a clean MemoryError instead of crashing VS Code /
# thrashing swap. This notebook builds cross-row features (lags/rolling), which
# can't be row-batched, so the guard is the protection here.
import os, sys
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *
install_memory_guard()


[memory_guard] hard cap 13.3G virtual on this kernel (total RAM 14.8G, 9.3G free now). Runaway allocations fail cleanly; bounded streaming keeps normal work well under this.


14298148864

In [2]:
import os
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

REGION        = variables.TARGET_REGION
ALL_REGIONS   = ["nsw", "qld", "vic", "sa"]
OTHER_REGIONS = [r for r in ALL_REGIONS if r != REGION]

PER_HOUR = 60 // variables.FEATURE_GRANULARITY_IN_MINUTES   # 12
PER_DAY  = 24 * PER_HOUR                                     # 288
PER_WEEK = 7 * PER_DAY                                       # 2016


def _fuel(df: pd.DataFrame, fuel: str, region: str) -> pd.Series:
    """Fuel-type MW series for a region, or zeros where that region has none."""
    col = f"{fuel}_mw_{region}"
    return df[col] if col in df.columns else pd.Series(0.0, index=df.index)

In [3]:
df = read_parquet_float32("../1_Dataset/Processed_data/3_generation_fuel.parquet")

df_core_columns = df.columns
df_base = df
df_base[:10]

Loading..:   0%|          | 0/9 [00:00<?, ?batch/s]

Loading..: 100%|██████████| 9/9 [00:00<00:00, 26.69batch/s]


,battery_charge_mw_nsw,battery_discharge_mw_nsw,biomass_mw_nsw,coal_mw_nsw,gas_mw_nsw,hydro_mw_nsw,solar_mw_nsw,wind_mw_nsw,battery_charge_mw_qld,battery_discharge_mw_qld,...,solar_mw_sa,wind_mw_sa,battery_charge_mw_vic,battery_discharge_mw_vic,biomass_mw_vic,coal_mw_vic,gas_mw_vic,hydro_mw_vic,solar_mw_vic,wind_mw_vic
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,0.0,0.0,0.0,5366.375488,0.214073,204.303329,0.107001,96.119987,0.0,0.0,...,0.0,284.732819,0.0,0.0,0.0,4666.302734,39.919998,0.115,0.0,273.751984
2018-01-01 00:10:00,0.0,0.0,0.0,5345.375000,0.220565,206.256805,0.107001,89.566231,0.0,0.0,...,0.0,259.189178,0.0,0.0,0.0,4664.306152,39.919998,0.115,0.0,272.872009
2018-01-01 00:15:00,0.0,0.0,0.0,5281.774902,0.220565,203.666397,0.107001,83.598763,0.0,0.0,...,0.0,251.515015,0.0,0.0,0.0,4667.756836,39.799999,0.115,0.0,267.476990
2018-01-01 00:20:00,0.0,0.0,0.0,5268.799805,0.220565,201.783142,0.107001,75.732513,0.0,0.0,...,0.0,240.667465,0.0,0.0,0.0,4670.655762,39.980000,0.115,0.0,262.239014
2018-01-01 00:25:00,0.0,0.0,0.0,5219.799805,0.220565,202.797577,0.107001,78.487503,0.0,0.0,...,0.0,240.368439,0.0,0.0,0.0,4671.654297,39.799999,0.115,0.0,251.347000
2018-01-01 00:30:00,0.0,0.0,0.0,5179.725098,0.220565,202.069626,0.107001,83.677498,0.0,0.0,...,0.0,251.897827,0.0,0.0,0.0,4664.079590,40.040001,0.115,0.0,253.347015
2018-01-01 00:35:00,0.0,0.0,0.0,5155.475098,0.220565,202.808304,0.107001,89.213730,0.0,0.0,...,0.0,249.425598,0.0,0.0,0.0,4671.666016,39.919998,0.115,0.0,260.583008
2018-01-01 00:40:00,0.0,0.0,0.0,5120.300293,0.220565,128.753586,0.107001,88.563751,0.0,0.0,...,0.0,233.768158,0.0,0.0,0.0,4664.589844,39.840000,0.115,0.0,273.989014
2018-01-01 00:45:00,0.0,0.0,0.0,5138.925293,0.220565,82.947647,0.107001,84.136230,0.0,0.0,...,0.0,213.520981,0.0,0.0,0.0,4666.273438,39.980000,0.115,0.0,284.519989


In [4]:
def _add_fuel_mix_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fuel-mix shares and penetration for the target region. High renewable
    penetration depresses prices while a coal/gas-dominated mix lifts the
    marginal cost. All generation is metered at the interval, so no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    coal = _fuel(df, "coal", R); gas = _fuel(df, "gas", R)
    hydro = _fuel(df, "hydro", R); solar = _fuel(df, "solar", R)
    wind = _fuel(df, "wind", R); bio = _fuel(df, "biomass", R)
    bd = _fuel(df, "battery_discharge", R); bc = _fuel(df, "battery_charge", R)

    thermal = coal + gas
    vre     = solar + wind
    renew   = vre + hydro + bio
    total   = thermal + hydro + solar + wind + bio + bd

    new_cols = {}
    new_cols[f"total_gen_{R}"]   = total.astype(np.float32)
    new_cols[f"thermal_gen_{R}"] = thermal.astype(np.float32)
    new_cols[f"vre_gen_{R}"]     = vre.astype(np.float32)
    new_cols[f"renew_pen_{R}"]   = (renew / (total + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"vre_pen_{R}"]     = (vre / (total + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"coal_share_{R}"]  = (coal / (total + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"gas_share_{R}"]   = (gas / (total + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"hydro_share_{R}"] = (hydro / (total + 1)).clip(0, 1).astype(np.float32)
    new_cols[f"battery_net_{R}"] = (bd - bc).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_fuel_mix_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,total_gen_nsw,thermal_gen_nsw,vre_gen_nsw,renew_pen_nsw,vre_pen_nsw,coal_share_nsw,gas_share_nsw,hydro_share_nsw,battery_net_nsw
Date,,,,,,,,,
2018-01-01 00:05:00,5667.119629,5366.589355,96.226990,0.053021,0.016977,0.946765,0.000038,0.036044,0.0
2018-01-01 00:10:00,5641.525879,5345.595703,89.673233,0.052446,0.015892,0.947337,0.000039,0.036554,0.0
2018-01-01 00:15:00,5569.367676,5281.995605,83.705765,0.051589,0.015027,0.948191,0.000040,0.036562,0.0
2018-01-01 00:20:00,5546.643066,5269.020508,75.839516,0.050043,0.013671,0.949737,0.000040,0.036373,0.0
2018-01-01 00:25:00,5501.412109,5220.020508,78.594505,0.051140,0.014284,0.948638,0.000040,0.036856,0.0
2018-01-01 00:30:00,5465.800293,5179.945801,83.784500,0.052289,0.015326,0.947488,0.000040,0.036963,0.0
2018-01-01 00:35:00,5447.824707,5155.695801,89.320732,0.053613,0.016393,0.946163,0.000040,0.037221,0.0
2018-01-01 00:40:00,5337.945312,5120.520996,88.670753,0.040724,0.016608,0.959047,0.000041,0.024116,0.0
2018-01-01 00:45:00,5306.336914,5139.145996,84.243233,0.031502,0.015873,0.968268,0.000042,0.015629,0.0


In [5]:
def _add_vre_dynamics_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Variable-renewable ramps and volatility for the target region. The evening
    solar down-ramp and wind lulls tighten the residual supply stack and are
    strong price predictors. Backward-looking differences / rolls only.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    solar = _fuel(df, "solar", R); wind = _fuel(df, "wind", R)
    vre = solar + wind

    new_cols = {}
    new_cols[f"vre_ramp_{R}_30m"]  = vre.diff(6).astype(np.float32)
    new_cols[f"vre_ramp_{R}_1h"]   = vre.diff(PER_HOUR).astype(np.float32)
    new_cols[f"solar_ramp_{R}_1h"] = solar.diff(PER_HOUR).astype(np.float32)
    new_cols[f"wind_mom_{R}_3h"]   = wind.diff(3 * PER_HOUR).astype(np.float32)

    new_cols[f"vre_rmean_{R}_1d"]  = vre.rolling(PER_DAY, min_periods=PER_HOUR).mean().astype(np.float32)
    new_cols[f"wind_rstd_{R}_1d"]  = wind.rolling(PER_DAY, min_periods=PER_HOUR).std().astype(np.float32)
    new_cols[f"solar_rmax_{R}_1d"] = solar.rolling(PER_DAY, min_periods=PER_HOUR).max().astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_vre_dynamics_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,vre_ramp_nsw_30m,vre_ramp_nsw_1h,solar_ramp_nsw_1h,wind_mom_nsw_3h,vre_rmean_nsw_1d,wind_rstd_nsw_1d,solar_rmax_nsw_1d
Date,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,-6.906258,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,-1.002480,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,0.537468,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def _add_fuel_lag_roll_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Backward-looking lags and daily rolling means for the main dispatchable and
    renewable fuels of the target region. Look-back only, no leakage.
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    LAGS = [1, PER_HOUR, PER_DAY, PER_WEEK]
    new_cols = {}
    for fuel in ["coal", "gas", "hydro", "solar", "wind"]:
        s = _fuel(df, fuel, R)
        for lag in LAGS:
            new_cols[f"{fuel}_{R}_lag_{lag}"] = s.shift(lag).astype(np.float32)
        new_cols[f"{fuel}_{R}_rmean_{PER_DAY}"] = (
            s.rolling(PER_DAY, min_periods=PER_HOUR).mean().astype(np.float32)
        )
    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_fuel_lag_roll_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,coal_nsw_lag_1,coal_nsw_lag_12,coal_nsw_lag_288,coal_nsw_lag_2016,coal_nsw_rmean_288,gas_nsw_lag_1,gas_nsw_lag_12,gas_nsw_lag_288,gas_nsw_lag_2016,gas_nsw_rmean_288,...,solar_nsw_lag_1,solar_nsw_lag_12,solar_nsw_lag_288,solar_nsw_lag_2016,solar_nsw_rmean_288,wind_nsw_lag_1,wind_nsw_lag_12,wind_nsw_lag_288,wind_nsw_lag_2016,wind_nsw_rmean_288
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,5366.375488,NaN,NaN,NaN,NaN,0.214073,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,96.119987,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,5345.375000,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,89.566231,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,5281.774902,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,83.598763,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,5268.799805,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,75.732513,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,5219.799805,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,78.487503,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,5179.725098,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,83.677498,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,5155.475098,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,89.213730,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,5120.300293,NaN,NaN,NaN,NaN,0.220565,NaN,NaN,NaN,NaN,...,0.107001,NaN,NaN,NaN,NaN,88.563751,NaN,NaN,NaN,NaN


In [7]:
def _add_national_fuel_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    NEM-wide fuel context and neighbouring-region renewables. National VRE
    output sets the interconnected price floor and spills across regions via
    the interconnectors. Contemporaneous / past values only — leakage-free.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    nat_solar = sum(_fuel(df, "solar", r) for r in ALL_REGIONS)
    nat_wind  = sum(_fuel(df, "wind", r)  for r in ALL_REGIONS)
    nat_total = sum(
        _fuel(df, fuel, r)
        for r in ALL_REGIONS
        for fuel in ["coal", "gas", "hydro", "solar", "wind", "biomass", "battery_discharge"]
    )
    new_cols["nem_solar_total"] = nat_solar.astype(np.float32)
    new_cols["nem_wind_total"]  = nat_wind.astype(np.float32)
    new_cols["nem_vre_pen"]     = ((nat_solar + nat_wind) / (nat_total + 1)).clip(0, 1).astype(np.float32)

    for r in OTHER_REGIONS:
        new_cols[f"neighbour_vre_{r}"] = (_fuel(df, "solar", r) + _fuel(df, "wind", r)).astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_national_fuel_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,nem_solar_total,nem_wind_total,nem_vre_pen,neighbour_vre_qld,neighbour_vre_vic,neighbour_vre_sa
Date,,,,,,
2018-01-01 00:05:00,2.057001,654.604797,0.035237,1.95,273.751984,284.732819
2018-01-01 00:10:00,1.407001,621.627441,0.033468,1.30,272.872009,259.189178
2018-01-01 00:15:00,1.407001,602.590759,0.032540,1.30,267.476990,251.515015
2018-01-01 00:20:00,1.407001,578.638977,0.031445,1.30,262.239014,240.667465
2018-01-01 00:25:00,1.407001,570.202942,0.031143,1.30,251.347000,240.368439
2018-01-01 00:30:00,1.407001,588.922363,0.032217,1.30,253.347015,251.897827
2018-01-01 00:35:00,1.407001,599.222351,0.032878,1.30,260.583008,249.425598
2018-01-01 00:40:00,1.407001,596.320923,0.032816,1.30,273.989014,233.768158
2018-01-01 00:45:00,1.407001,582.177185,0.032158,1.30,284.519989,213.520981


In [8]:
def _add_thermal_availability_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Thermal (coal / gas) availability shocks for the target region. A sudden
    fall in coal or gas output is the classic trigger for extreme price spikes
    (generator trips / forced outages). Installed capacity is proxied by the
    trailing rolling max of output -- region-agnostic, so utilisation and
    sustained-low-availability signals adapt to each region without a hard-coded
    plant list. All look-backs are backward-only -> leakage-free.

    (Restores the coal outage / low-availability logic from the original
    single-region notebook, generalised to the multi-region schema.)
    Returns only the new columns to avoid copying the full base frame.
    """
    R = REGION
    new_cols = {}
    for fuel in ["coal", "gas"]:
        s = _fuel(df, fuel, R)
        cap = s.rolling(PER_WEEK, min_periods=PER_DAY).max()   # capacity proxy

        # Rate-of-change: rapid negative moves signal a trip.
        new_cols[f"{fuel}_chg_{R}_1h"] = s.diff(PER_HOUR).astype(np.float32)
        new_cols[f"{fuel}_chg_{R}_6h"] = s.diff(6 * PER_HOUR).astype(np.float32)

        # Trip severity: how far output fell below its level 2h ago (MW).
        drop_2h = s.shift(2 * PER_HOUR) - s.rolling(2 * PER_HOUR, min_periods=PER_HOUR).min()
        new_cols[f"{fuel}_drop_{R}_2h_mw"] = drop_2h.clip(lower=0).astype(np.float32)

        # Utilisation vs trailing capacity proxy (low = spare capacity offline).
        new_cols[f"{fuel}_util_{R}"] = (s / (cap + 1)).clip(0, 1.05).astype(np.float32)

        # Sustained low availability (< 55% of capacity proxy) over 1d / 1w.
        low = (s < 0.55 * cap).astype(np.float32)
        new_cols[f"{fuel}_low_count_{R}_1d"] = low.rolling(PER_DAY, min_periods=PER_HOUR).sum().astype(np.float32)
        new_cols[f"{fuel}_low_count_{R}_1w"] = low.rolling(PER_WEEK, min_periods=PER_DAY).sum().astype(np.float32)

    return pd.DataFrame(new_cols, index=df.index)


new_df = _add_thermal_availability_features(df_base)
df = pd.concat([df, new_df], axis=1)
new_df[:10]


,coal_chg_nsw_1h,coal_chg_nsw_6h,coal_drop_nsw_2h_mw,coal_util_nsw,coal_low_count_nsw_1d,coal_low_count_nsw_1w,gas_chg_nsw_1h,gas_chg_nsw_6h,gas_drop_nsw_2h_mw,gas_util_nsw,gas_low_count_nsw_1d,gas_low_count_nsw_1w
Date,,,,,,,,,,,,
2018-01-01 00:05:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:10:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:20:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-01 00:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
# Retain core columns: metered generation-by-fuel is known at the interval t
# -> leakage-free (already exposed via the fuel-mix features).
print("Total features:", df.shape[1])
df.to_parquet("../2_Features_build/Feature_data/3_generation_fuel.parquet")
df.shape

Total features: 91


(893664, 91)

In [10]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()


[release_memory] cleared 15 variable(s); kernel rss 0.64G, 8.6G RAM free now
